In [1]:
import sys
from pathlib import Path

import torch
import modelopt.torch.opt as mto
import modelopt.torch.quantization as mtq

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "pyfiles"))
sys.path.insert(0, str(ROOT / "pyfiles" / "qat_modelopt"))

from quantize import get_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
# 1. Rebuild the model architecture
model = get_model("/home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_42/best.pth", num_classes=100).to(device)

# 2. Restore quantization structure from mostate
mostate = torch.load("/home/pf4636/code/resnet/quantized_resnets/checkpoints/qat/int4_in8b/seed_42/qat_modelopt_best_mostate.pt", map_location=device)
mto.restore_from_modelopt_state(model, mostate)

# 3. Load weights from pth
ckpt = torch.load("/home/pf4636/code/resnet/quantized_resnets/checkpoints/qat/int4_in8b/seed_42/qat_modelopt_best.pth", map_location=device)
model.load_state_dict(ckpt["model"])

# 4. Verify quantizers
all_quantizers = [(name, mod) for name, mod in model.named_modules()
                  if isinstance(mod, mtq.nn.TensorQuantizer)]
print(f"Total quantizers: {len(all_quantizers)}")

for name, mod in all_quantizers:
    if "weight" in name:
        print(f"{name}: num_bits={mod.num_bits}, amax={mod.amax}")